# 🧊 Sea Ice SAR Segmentation — Google Colab Training

**Model:** 8-Module Pipeline (CLIP ViT-L/14 + DepthAnything V2 + SAM + Cross-Attention CoT)

**Task:** 6-class sea ice segmentation from SAR imagery

**Classes:** Young Ice · First Year Ice · Floating Ice · Glaciers · Icebergs · Old Ice

---
**Before you start:**
1. `Runtime → Change runtime type → GPU` (T4 is free, A100/V100 via Colab Pro)
2. Run cells **in order** from top to bottom
3. Checkpoints auto-save to Google Drive every 1 000 steps

> ⏱ Estimated training time: ~3–4 h (T4, 50 epochs, no SAM) | ~6–8 h (A100, with SAM)

## 1. 🖥️ Check GPU

In [ ]:
import subprocess, sys, os

# GPU info
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError('❌ No GPU detected! Go to Runtime → Change runtime type → GPU')
print(result.stdout)

import torch
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.version.cuda}')
print(f'Device   : {torch.cuda.get_device_name(0)}')
print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'\n✅ GPU ready: {GPU_NAME} ({VRAM_GB:.0f} GB)')

## 2. 📂 Mount Google Drive (for persistent checkpoints)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/sea_ice_seg'
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/outputs', exist_ok=True)
print(f'✅ Drive mounted. Project folder: {DRIVE_DIR}')

## 3. 📥 Clone Repository

In [ ]:
import os

REPO_URL  = 'https://github.com/prakhar443/sea_ice_seg.git'
BRANCH    = 'claude/laughing-thompson-AhCX7'  # branch that contains all fixes
REPO_DIR  = '/content/sea_ice_seg'

if os.path.exists(REPO_DIR):
    print('Repo already cloned. Pulling latest...')
    !git -C {REPO_DIR} fetch origin {BRANCH}
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} reset --hard origin/{BRANCH}
else:
    print(f'Cloning branch {BRANCH}...')
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f'\n✅ Working directory: {os.getcwd()}')
!ls -la


## 4. 📦 Install Dependencies

> Takes ~3–5 minutes on first run. Subsequent runs are faster.

In [ ]:
print('Installing core dependencies...')

# torchao 0.10.0 (Colab default) is incompatible with current peft (requires >=0.16.0).
# Upgrade it first so peft's import_utils check passes.
!pip install -q 'torchao>=0.16.0'

!pip install -q \
    'transformers>=4.40.0' \
    'peft>=0.10.0' \
    'accelerate>=0.27.0' \
    'einops>=0.7.0' \
    'timm>=0.9.0' \
    'albumentations>=2.0.0' \
    openpyxl \
    pandas \
    scipy \
    scikit-learn \
    tqdm \
    wandb

print('\nInstalling Segment Anything Model (SAM)...')
!pip install -q git+https://github.com/facebookresearch/segment-anything.git

# Print key versions so we can spot future conflicts early
import albumentations as A, peft, torchao
albu_major = int(A.__version__.split('.')[0])
print(f'\nalbumentations : {A.__version__}  (API: {"2.x size=" if albu_major >= 2 else "1.x h/w"})')
print(f'peft           : {peft.__version__}')
print(f'torchao        : {torchao.__version__}')

print('\nVerifying key packages...')
import importlib
for pkg in ['transformers', 'peft', 'albumentations', 'segment_anything', 'einops', 'timm']:
    try:
        m = importlib.import_module(pkg)
        ver = getattr(m, '__version__', 'ok')
        print(f'  ✅ {pkg} {ver}')
    except ImportError:
        print(f'  ❌ {pkg} NOT FOUND')

print('\n✅ All dependencies installed!')


## 5. ⚙️ Auto-configure Hyperparameters for Your GPU

Automatically picks the right batch size, memory-efficient settings, and whether to use SAM.

In [ ]:
import torch

VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
GPU_NAME = torch.cuda.get_device_name(0)

# ─── Auto-select settings based on available VRAM ────────────────────────────
# DECODER_BASE = U-Net width. Bigger = sharper masks + more VRAM. Drop it (or
# lower IMAGE_SIZE) if you hit CUDA OOM.
if VRAM_GB >= 38:   # A100 (40 GB)
    BATCH_SIZE     = 4
    # USE_SAM=False: SAM ViT-H is FROZEN and prompt-driven, which BYPASSES the
    # trainable ImageUNetDecoder (and its deep-supervision aux head). Every
    # fix in this pipeline — mask/image alignment, decoder_lr, aux loss —
    # targets the U-Net, so we must keep the U-Net path active. SAM here caps
    # mIoU and forces aux=0.000. The A100 just lets us run it bigger/faster.
    USE_SAM        = False
    GRAD_ACCUM     = 4      # effective batch = 16
    NUM_WORKERS    = 4
    IMAGE_SIZE     = (512, 512)
    LLM_BACKEND    = 'cross_attn_only'
    DECODER_BASE   = 48
elif VRAM_GB >= 14:  # T4 / V100 (15–16 GB)
    BATCH_SIZE     = 2
    USE_SAM        = False  # SAM ViT-H needs ~8 GB extra
    GRAD_ACCUM     = 8      # effective batch = 16
    NUM_WORKERS    = 2
    IMAGE_SIZE     = (512, 512)
    LLM_BACKEND    = 'cross_attn_only'
    DECODER_BASE   = 32
else:               # < 14 GB (smaller GPUs)
    BATCH_SIZE     = 1
    USE_SAM        = False
    GRAD_ACCUM     = 16     # effective batch = 16
    NUM_WORKERS    = 2
    IMAGE_SIZE     = (384, 384)
    LLM_BACKEND    = 'cross_attn_only'
    DECODER_BASE   = 16

# ─── Training settings ────────────────────────────────────────────────────────
# Conservative LRs — avoids NaN loss / gradient explosion on small datasets
EPOCHS         = 50          # early stopping will stop sooner if val mIoU plateaus
LEARNING_RATE  = 5e-5        # base lr (was 2e-4 — too aggressive for ~420 samples)
LORA_LR        = 2e-4        # LoRA adapters lr (was 1e-3)
CLS_HEAD_LR    = 1e-4        # classification head lr (was 5e-4)
EARLY_PATIENCE = 12          # stop after this many evals with no improvement

USE_FP16      = True
USE_BF16      = False   # overridden below for A100
USE_WANDB     = False

# A100 has native BF16: same exponent range as FP32 → no overflow, no NaN cascade.
# FP16 saturates at 65504; the aux_head Conv can reach this after ~10 epochs
# and the subsequent focal-loss backward then produces Inf gradients → NaN weights.
if VRAM_GB >= 38:
    USE_FP16 = False
    USE_BF16 = True     # Set True and add your API key for W&B logging
WANDB_PROJECT = 'sea-ice-seg'

DATA_ROOT     = '/content/sea_ice_seg/dataset'
OUTPUT_DIR    = '/content/sea_ice_seg/outputs'
DRIVE_OUTPUT  = '/content/drive/MyDrive/sea_ice_seg/outputs'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('═' * 55)
print(f'  GPU          : {GPU_NAME} ({VRAM_GB:.0f} GB)')
print(f'  Batch size   : {BATCH_SIZE}  (effective = {BATCH_SIZE * GRAD_ACCUM})')
print(f'  Grad accum   : {GRAD_ACCUM} steps')
print(f'  Use SAM      : {USE_SAM}')
print(f'  FP16         : {USE_FP16}')
print(f'  Epochs       : {EPOCHS}  (+ early stop patience={EARLY_PATIENCE})')
print(f'  Base LR      : {LEARNING_RATE}  LoRA LR: {LORA_LR}')
print(f'  LLM backend  : {LLM_BACKEND}')
print(f'  Image size   : {IMAGE_SIZE}')
print(f'  Decoder      : U-Net (base={DECODER_BASE})')
print(f'  Output dir   : {OUTPUT_DIR}')
print('═' * 55)

## 6. 🔍 Verify Dataset Structure

In [ ]:
import os
from pathlib import Path

data_root = Path(DATA_ROOT)
ice_classes = ['Young Ice', 'First Year Ice', 'Floating Ice',
               'Glaciers', 'Icebergs', 'Old Ice']

print('Dataset structure:')
print('=' * 60)
total_images = 0
total_masks  = 0
for cls in ice_classes:
    cls_dir   = data_root / cls
    img_dir   = cls_dir / 'images'
    mask_dir  = cls_dir / 'masks'
    desc_dir  = cls_dir / 'descriptions'

    n_imgs  = len(list(img_dir.glob('*')))  if img_dir.exists()  else 0
    n_masks = len(list(mask_dir.glob('*'))) if mask_dir.exists() else 0
    n_desc  = len(list(desc_dir.glob('*.xlsx'))) if desc_dir.exists() else 0

    status = '✅' if (n_imgs > 0 and n_masks > 0) else '❌'
    print(f'{status} {cls:<20} images={n_imgs:>4}  masks={n_masks:>4}  xlsx={n_desc}')
    total_images += n_imgs
    total_masks  += n_masks

print('=' * 60)
print(f'Total: {total_images} images  |  {total_masks} masks')
print()
if total_images == 0:
    print('❌ No images found! Make sure the dataset was included in the repo.')
else:
    print(f'✅ Dataset ready — {total_images} samples across {len(ice_classes)} classes')

## 7. 🖼️ Visualise Sample Images

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import numpy as np
from pathlib import Path

data_root   = Path(DATA_ROOT)
ice_classes = ['Young Ice', 'First Year Ice', 'Floating Ice',
               'Glaciers', 'Icebergs', 'Old Ice']
colors = ['#4FC3F7','#0288D1','#80CBC4','#A5D6A7','#CE93D8','#FFB74D']

fig, axes = plt.subplots(2, 6, figsize=(22, 8))
fig.suptitle('Sample SAR Images (top) and Masks (bottom)', fontsize=14, fontweight='bold')

for col, cls in enumerate(ice_classes):
    img_dir  = data_root / cls / 'images'
    mask_dir = data_root / cls / 'masks'
    imgs = sorted(img_dir.glob('*.jpg'))
    if not imgs:
        continue

    # Pick first image
    img_path  = imgs[0]
    stem      = img_path.stem.rstrip('_')
    mask_path = mask_dir / (stem + '_scat.jpg')

    img  = Image.open(img_path).convert('RGB')
    axes[0, col].imshow(img, cmap='gray')
    axes[0, col].set_title(cls, fontsize=8, color=colors[col], fontweight='bold')
    axes[0, col].axis('off')

    if mask_path.exists():
        mask = Image.open(mask_path).convert('L')
        axes[1, col].imshow(mask, cmap='gray')
    else:
        axes[1, col].text(0.5, 0.5, 'No mask', ha='center', va='center',
                          transform=axes[1, col].transAxes)
    axes[1, col].axis('off')

plt.tight_layout()
plt.savefig('/content/sample_images.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Sample visualisation saved to /content/sample_images.png')

## 7b. 🔬 Mask Diagnostic — Raw Scattering Map vs Binarized Target

> **Important:** the `_scat` masks are *continuous scattering maps* (130–160 grey
> levels), not clean binary labels, and have a different aspect ratio than the
> images (≈138×187 vs 256×256). This cell shows the raw mask, the **Otsu-binarized
> target** the model actually trains on, and the overlay on the SAR image.
>
> **Check that the binarized region plausibly corresponds to the ice in the image.**
> If it does → training should now learn real segmentation. If it clearly does
> not → the masks aren't usable segmentation ground truth and mIoU will stay low
> no matter what (a data problem, not a code problem).

In [ ]:
import sys
sys.path.insert(0, '/content/sea_ice_seg')
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from data.dataset import binarize_mask, letterbox_to_square

# This cell runs before the config-patch cell, so don't depend on `cfg`.
BINARIZE_MODE = 'otsu'   # must match cfg.data.mask_binarize used in training

data_root   = Path(DATA_ROOT)
ice_classes = ['Young Ice', 'First Year Ice', 'Floating Ice',
               'Glaciers', 'Icebergs', 'Old Ice']

fig, axes = plt.subplots(4, 6, figsize=(24, 16))
row_titles = ['SAR Image', 'Raw _scat Mask', 'Otsu Binary Target', 'Overlay (img+target)']
for r, t in enumerate(row_titles):
    axes[r, 0].set_ylabel(t, fontsize=11, fontweight='bold')

print(f'{"class":<16}{"raw fg(>127)":<14}{"otsu fg":<12}')
print('-' * 44)

for col, cls in enumerate(ice_classes):
    img_dir  = data_root / cls / 'images'
    mask_dir = data_root / cls / 'masks'
    imgs = sorted(img_dir.glob('*.jpg'))
    if not imgs:
        continue
    img_path  = imgs[0]
    stem      = img_path.stem.rstrip('_')
    mask_path = mask_dir / (stem + '_scat.jpg')
    if not mask_path.exists():
        cands = list(mask_dir.glob(stem + '*'))
        mask_path = cands[0] if cands else None
    if mask_path is None:
        continue

    img_gray  = np.array(Image.open(img_path).convert('L'))
    mask_gray = np.array(Image.open(mask_path).convert('L'))

    # Same pipeline the dataloader uses (mask_resize_mode = 'stretch'):
    # image and mask cover the SAME scene at different sampling resolutions,
    # so the mask is stretched onto the image's extent to keep them aligned.
    mask_bin  = binarize_mask(mask_gray, mode=BINARIZE_MODE)
    img_lb    = img_gray
    mask_lb   = np.array(Image.fromarray(mask_bin).resize(
                    (img_gray.shape[1], img_gray.shape[0]), Image.NEAREST))

    raw_fg  = (mask_gray > 127).mean() * 100
    otsu_fg = mask_bin.mean() * 100
    print(f'{cls:<16}{raw_fg:<14.1f}{otsu_fg:<12.1f}')

    axes[0, col].imshow(img_gray, cmap='gray');                axes[0, col].set_title(cls, fontsize=9)
    axes[1, col].imshow(mask_gray, cmap='viridis')             # raw continuous map
    axes[2, col].imshow(mask_lb, cmap='gray')                  # binarized target
    axes[3, col].imshow(img_lb, cmap='gray')
    axes[3, col].imshow(mask_lb, cmap='Reds', alpha=0.45)      # overlay
    for r in range(4):
        axes[r, col].set_xticks([]); axes[r, col].set_yticks([])

plt.tight_layout()
plt.savefig('/content/mask_diagnostic.png', dpi=110, bbox_inches='tight')
plt.show()
print('\n👉 Look at row 4 (overlay): does the red target region sit on the ice in the SAR image?')
print('   If yes → the new Otsu + Focal/Tversky setup should learn it.')
print('   If the red region looks random vs the image → masks are not valid GT (data issue).')

## 8. 🏋️ (Optional) Download SAM Checkpoint

SAM ViT-H improves segmentation accuracy but requires **~8 GB extra VRAM**.  
Skip this cell if your GPU has ≤ 16 GB or if `USE_SAM = False`.

In [ ]:
import os

SAM_CHECKPOINT = '/content/sea_ice_seg/checkpoints/sam_vit_h_4b8939.pth'
SAM_URL        = 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth'
DRIVE_SAM      = '/content/drive/MyDrive/sea_ice_seg/checkpoints/sam_vit_h_4b8939.pth'
SAM_MIN_BYTES  = 2_400_000_000   # sam_vit_h is ~2.56 GB; smaller => truncated/corrupt

os.makedirs('/content/sea_ice_seg/checkpoints', exist_ok=True)

def _sam_is_valid(path):
    """Valid only if it exists, is full-size, and torch can actually open it."""
    if not os.path.exists(path):
        return False
    if os.path.getsize(path) < SAM_MIN_BYTES:
        print(f'  ⚠️  {path} is only {os.path.getsize(path)/1e9:.2f} GB — truncated.')
        return False
    try:
        import torch
        torch.load(path, map_location='cpu', weights_only=False)
        return True
    except Exception as e:
        print(f'  ⚠️  {path} failed to load ({type(e).__name__}) — corrupt.')
        return False

if not USE_SAM:
    print('USE_SAM = False → skipping SAM download (lightweight decoder will be used)')
elif _sam_is_valid(SAM_CHECKPOINT):
    print(f'✅ SAM checkpoint already present and valid: {SAM_CHECKPOINT}')
else:
    # Drop any corrupt local copy first
    if os.path.exists(SAM_CHECKPOINT):
        os.remove(SAM_CHECKPOINT)

    # Use the Drive cache only if it is itself valid
    if _sam_is_valid(DRIVE_SAM):
        print('Copying valid SAM checkpoint from Drive cache...')
        !cp {DRIVE_SAM} {SAM_CHECKPOINT}
    else:
        if os.path.exists(DRIVE_SAM):
            print('Drive cache is corrupt — removing and re-downloading.')
            os.remove(DRIVE_SAM)
        print('Downloading SAM ViT-H checkpoint (~2.5 GB)... (~5 min on Colab)')
        # -c resumes a partial file rather than appending to a broken one
        !wget -q --show-progress -c -O {SAM_CHECKPOINT} {SAM_URL}

    # Final integrity gate — only cache to Drive if the file is genuinely valid
    if _sam_is_valid(SAM_CHECKPOINT):
        os.makedirs(os.path.dirname(DRIVE_SAM), exist_ok=True)
        !cp {SAM_CHECKPOINT} {DRIVE_SAM}
        print('✅ SAM checkpoint verified and cached to Drive')
    else:
        print('❌ SAM checkpoint still invalid after download.')
        print('   No problem — the model auto-falls back to the lightweight decoder,')
        print('   so you can keep going. Or set USE_SAM = False in Cell 5 and re-run.')

## 9. 🔧 Patch Config for Colab

Overrides `config.py` settings with the auto-selected Colab values.

In [ ]:
import sys
sys.path.insert(0, '/content/sea_ice_seg')

from config import cfg

# ── Data config ───────────────────────────────────────────────────────────────
cfg.data.data_root  = DATA_ROOT
cfg.data.image_size = IMAGE_SIZE

# ── Model config ──────────────────────────────────────────────────────────────
cfg.model.llm_backend     = LLM_BACKEND
cfg.model.sam_checkpoint  = SAM_CHECKPOINT
cfg.model.decoder_type          = 'unet'          # full-res U-Net (sharp masks)
cfg.model.decoder_base_channels = DECODER_BASE    # VRAM-aware width (Cell 5)

# ── Train config ──────────────────────────────────────────────────────────────
cfg.train.batch_size       = BATCH_SIZE
cfg.train.grad_accum_steps = GRAD_ACCUM
cfg.train.epochs           = EPOCHS
cfg.train.lr               = LEARNING_RATE
cfg.train.lora_lr          = LORA_LR       # conservative LoRA lr
cfg.train.cls_head_lr      = CLS_HEAD_LR   # conservative classifier head lr
cfg.train.fp16             = USE_FP16
cfg.train.bf16             = USE_BF16
cfg.train.num_workers      = NUM_WORKERS
cfg.train.output_dir       = OUTPUT_DIR
cfg.train.use_wandb        = USE_WANDB
cfg.train.wandb_project    = WANDB_PROJECT
cfg.train.early_stop_patience = EARLY_PATIENCE
# Eval every ~1 epoch on small dataset (600 samples / batch_size / grad_accum)
cfg.train.eval_every       = max(10, len(list(__import__('pathlib').Path(DATA_ROOT).rglob('images/*.jpg'))) // BATCH_SIZE // GRAD_ACCUM)
cfg.train.save_every       = cfg.train.eval_every * 5

print('Config patched for Colab:')
print(f'  data_root        = {cfg.data.data_root}')
print(f'  image_size       = {cfg.data.image_size}')
print(f'  batch_size       = {cfg.train.batch_size}')
print(f'  grad_accum       = {cfg.train.grad_accum_steps}')
print(f'  epochs           = {cfg.train.epochs}')
print(f'  lr               = {cfg.train.lr}  (lora={cfg.train.lora_lr}, cls={cfg.train.cls_head_lr}, dec={cfg.train.decoder_lr})')
print(f'  decoder          = {cfg.model.decoder_type} (base={cfg.model.decoder_base_channels})')
print(f'  precision        = {"BF16" if cfg.train.bf16 else "FP16" if cfg.train.fp16 else "FP32"}')
print(f'  eval_every       = {cfg.train.eval_every} steps (~1 epoch)')
print(f'  early_patience   = {cfg.train.early_stop_patience} evals')
print(f'  num_workers      = {cfg.train.num_workers}')
print(f'  llm_backend      = {cfg.model.llm_backend}')
print(f'  use_sam          = {USE_SAM}')
print(f'  use_wandb        = {cfg.train.use_wandb}')
print('\n✅ Config ready')

## 10. 📊 Build Dataloaders & Inspect

In [ ]:
import sys
sys.path.insert(0, '/content/sea_ice_seg')

from data.dataset import build_dataloaders

train_loader, val_loader, test_loader = build_dataloaders(cfg.data, cfg.train)

# Peek at one batch
batch = next(iter(train_loader))
print('\nBatch shapes:')
print(f'  image  : {batch["image"].shape}   dtype={batch["image"].dtype}')
print(f'  mask   : {batch["mask"].shape}    dtype={batch["mask"].dtype}')
print(f'  label  : {batch["label"]}')
print(f'  short  : {batch["short_desc"][0][:60]}...')
print(f'  long   : {batch["long_desc"][0][:80]}...')
print(f'\n✅ Dataloaders ready')

## 11. 🏗️ Build Model

In [ ]:
import torch, sys
sys.path.insert(0, '/content/sea_ice_seg')

from models.pipeline import SeaIceSegmentationPipeline

device = cfg.train.device
print(f'Building model on {device}...')
print('  CLIP ViT-L/14 + LoRA  — downloading from HuggingFace if needed...')
print('  DepthAnything V2       — downloading from HuggingFace if needed...')
print('  (first run may take a few minutes)\n')

model = SeaIceSegmentationPipeline(cfg.model, use_sam=USE_SAM).to(device)

total_params    = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params   = total_params - trainable_params

print(f'\nModel summary:')
print(f'  Total parameters    : {total_params:>12,}')
print(f'  Trainable (LoRA+head): {trainable_params:>12,}')
print(f'  Frozen (CLIP/Depth) : {frozen_params:>12,}')

vram_used = torch.cuda.memory_allocated(0) / 1e9
print(f'\n  VRAM used after model load: {vram_used:.2f} GB / {VRAM_GB:.0f} GB')
print(f'\n✅ Model ready on {device}')

## 12. 🚀 Train the Model

> Checkpoints are saved to `OUTPUT_DIR` every `save_every` steps.  
> **mIoU** and **F1** are tracked and reported **separately** (standard practice).  
> Segmentation is the primary task, so `best_model.pth` and early stopping are  
> driven by **val mIoU**; the best **F1** is logged alongside for reference.

> ⚠️ **If you just `git pull`-ed new code: `Runtime → Restart runtime` first.**  
> Colab caches imported modules — without a restart you'll keep running the old
> `losses.py` / `pipeline.py`. The tell-tale sign of stale code is `aux=0.000`
> in the progress bar (deep supervision not active).

In [ ]:
# ── Clear stale module cache ─────────────────────────────────────────────────
# Python (and Colab) keep imported modules in sys.modules. After a git pull
# you MUST either restart the runtime OR run this block to force a fresh
# import of every project file (config, train, models, utils, data).
import sys as _sys
_our_pkgs = ['config', 'train', 'data', 'models', 'utils']
_stale = [k for k in list(_sys.modules.keys())
          if any(k == p or k.startswith(p + '.') for p in _our_pkgs)]
for _k in _stale:
    del _sys.modules[_k]
print(f'Cleared {len(_stale)} cached module(s) — fresh imports will load.')
del _sys, _our_pkgs, _stale, _k

import os, sys, math, random
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.amp import GradScaler, autocast
from tqdm.notebook import tqdm
sys.path.insert(0, '/content/sea_ice_seg')

from utils.losses  import SeaIceLoss
from utils.metrics import MetricAccumulator
from train import set_seed, get_lr_scheduler, save_checkpoint, validate, cleanup_old_checkpoints

set_seed(cfg.train.seed)

output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

# ── Optimizer ─────────────────────────────────────────────────────────────────
param_groups = model.get_param_groups(cfg.model, cfg.train)
optimizer    = torch.optim.AdamW(
    param_groups,
    lr=cfg.train.lr,
    weight_decay=cfg.train.weight_decay,
    betas=cfg.train.betas,
)

# ── Scheduler ─────────────────────────────────────────────────────────────────
steps_per_epoch  = len(train_loader) // cfg.train.grad_accum_steps
total_steps      = steps_per_epoch * cfg.train.epochs
scheduler        = get_lr_scheduler(optimizer, cfg.train, total_steps)

# ── Mixed precision ───────────────────────────────────────────────────────────
# BF16 has FP32's exponent range — no underflow/overflow, no GradScaler needed.
# FP16 still needs the scaler (underflow guard). Never use scaler with BF16.
_use_bf16 = cfg.train.bf16
_use_fp16 = cfg.train.fp16 and not _use_bf16
scaler    = GradScaler('cuda') if _use_fp16 else None

# ── Loss ──────────────────────────────────────────────────────────────────────
criterion = SeaIceLoss(cfg.train).to(device)

# ── (Optional) Resume from checkpoint ────────────────────────────────────────
# To continue from a previous run set this to the saved best_model.pth path.
# The stored 'best_metric' is the best val mIoU.
RESUME_CHECKPOINT = None   # e.g. '/content/drive/MyDrive/sea_ice_seg/outputs/best_model.pth'

start_epoch    = 0
start_step     = 0
best_metric    = 0.0       # best val mIoU

if RESUME_CHECKPOINT and os.path.exists(RESUME_CHECKPOINT):
    from train import load_checkpoint
    ckpt        = load_checkpoint(model, optimizer, scheduler, scaler,
                                  Path(RESUME_CHECKPOINT), device)
    start_epoch = ckpt['epoch']
    start_step  = ckpt['step']
    best_metric = ckpt['best_metric']

# Sanity: confirm deep supervision is wired (catches stale cached modules).
import inspect
_fwd = inspect.signature(model.forward).parameters
assert 'return_aux' in _fwd, (
    'model.forward has no return_aux — you are running a STALE cached pipeline. '
    'Do Runtime → Restart runtime, then re-run from the top.'
)

print(f'Training for {cfg.train.epochs} epochs, {total_steps} total optimizer steps')
print(f'Resuming: epoch={start_epoch}, step={start_step}, best_mIoU={best_metric:.4f}')
print(f'Loss weights: lambda_mask={cfg.train.lambda_mask}  '
      f'lambda_cls={cfg.train.lambda_cls}  lambda_aux={cfg.train.lambda_aux}  '
      f'grad_clip={getattr(cfg.train, "grad_clip_norm", 1.0)}')
print('Starting training...\n')

In [ ]:
# ─── Training loop ────────────────────────────────────────────────────────────
from pathlib import Path as _Path
import time
from IPython.display import clear_output

train_history = {'loss': [], 'miou': [], 'f1': [], 'step': []}
val_history   = {'miou': [], 'f1': [], 'step': []}

global_step        = start_step
# mIoU and F1 tracked separately. best_model.pth + early stopping use mIoU
# (segmentation is the primary task); best F1 is logged for reference.
best_miou          = best_metric
best_f1            = 0.0
grad_accum         = cfg.train.grad_accum_steps
use_amp            = cfg.train.fp16 or cfg.train.bf16
amp_dtype          = torch.bfloat16 if cfg.train.bf16 else torch.float16
log_every          = cfg.train.log_every
eval_every         = cfg.train.eval_every
save_every         = cfg.train.save_every
patience           = cfg.train.early_stop_patience
clip_norm          = getattr(cfg.train, 'grad_clip_norm', 1.0)
patience_counter   = 0
nan_skip_count     = 0

wandb_run = None
if USE_WANDB:
    import wandb
    wandb_run = wandb.init(project=WANDB_PROJECT, config=vars(cfg))

stop_training = False

for epoch in range(start_epoch, cfg.train.epochs):
    if stop_training:
        break

    model.train()
    model.temporal.reset_all()   # discard stale epoch-N embeddings before epoch N+1
    epoch_metrics = MetricAccumulator()
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{cfg.train.epochs}', leave=True)

    for batch_idx, batch in enumerate(pbar):
        images       = batch['image'].to(device)
        masks        = batch['mask'].to(device)
        labels       = batch['label'].to(device)
        descriptions = batch['long_desc']

        # sequence_id = ice-class folder name (e.g. "Young Ice"), not "images".
        # path layout: .../dataset/<class>/images/<file>  → parent.parent.name
        seq_ids = [str(_Path(p).parent.parent.name) for p in batch['image_path']]

        images_np = None
        if USE_SAM:
            images_np = [
                (img.permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
                for img in images
            ]

        with autocast('cuda', dtype=amp_dtype, enabled=use_amp):
            outputs = model(
                images=images,
                descriptions=descriptions,
                images_np=images_np,
                sequence_ids=seq_ids,
                return_aux=True,         # deep supervision from U-Net aux head
            )
            loss_dict = criterion(outputs, {'mask': masks, 'label': labels})
            loss = loss_dict['loss'] / grad_accum

        # ── NaN guard ────────────────────────────────────────────────────────
        if not torch.isfinite(loss):
            nan_skip_count += 1
            optimizer.zero_grad()
            # Clear temporal memory bank: a NaN logit stored there poisons
            # every future batch for the same sequence_id.
            model.temporal.reset_all()
            if nan_skip_count % 10 == 1:
                print(f'  ⚠️  NaN/Inf loss at step {global_step} '
                      f'(skipped {nan_skip_count} total) — temporal bank cleared. '
                      f'mask={loss_dict["loss_mask"].item():.4f}  '
                      f'cls={loss_dict["loss_cls"].item():.4f}')
            pbar.set_postfix({'loss': 'NaN-skip', 'skips': nan_skip_count})
            continue

        if scaler:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        if (batch_idx + 1) % grad_accum == 0:
            if scaler:
                scaler.unscale_(optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
            if torch.isfinite(grad_norm):
                if scaler:
                    scaler.step(optimizer)
                else:
                    optimizer.step()
                global_step += 1
            else:
                # NaN/Inf gradient despite finite loss — backward overflowed
                # (classic FP16 large-logit issue). Skip step, clear temporal bank.
                nan_skip_count += 1
                model.temporal.reset_all()
                if nan_skip_count % 5 == 1:
                    print(f'  ⚠️  NaN gradient norm at step {global_step} '
                          f'(skipped {nan_skip_count} total) — weights NOT updated')
            if scaler:
                scaler.update()
            optimizer.zero_grad()
            scheduler.step()

            raw_loss = loss_dict['loss'].item()
            epoch_metrics.update(
                outputs=outputs,
                targets={'mask': masks, 'label': labels},
                loss=raw_loss,
            )

            lr = optimizer.param_groups[0]['lr']
            pbar.set_postfix({
                'loss': f'{raw_loss:.4f}',
                'mask': f'{loss_dict["loss_mask"].item():.3f}',
                'cls':  f'{loss_dict["loss_cls"].item():.3f}',
                'aux':  f'{loss_dict["loss_aux"].item():.3f}',
                'lr':   f'{lr:.1e}',
            })

            # ── Validation + early stopping (driven by mIoU) ──────────────────
            if global_step % eval_every == 0 and global_step > 0:
                val_metrics = validate(model, val_loader, criterion, device)
                miou = val_metrics['mean_iou']
                f1   = val_metrics.get('weighted_f1', 0.0)
                if f1 != f1: f1 = 0.0   # guard NaN

                if f1 > best_f1:
                    best_f1 = f1

                print(f'  Val @ step {global_step}: '
                      f'mIoU={miou:.4f} (best={best_miou:.4f})  '
                      f'F1={f1:.4f} (best={best_f1:.4f})  '
                      f'patience {patience_counter}/{patience}')
                val_history['miou'].append(miou)
                val_history['f1'].append(f1)
                val_history['step'].append(global_step)

                if wandb_run:
                    wandb_run.log({f'val/{k}': v for k, v in val_metrics.items()
                                   if isinstance(v, (int, float))}, step=global_step)

                # Best model + early stopping: mIoU (primary segmentation metric)
                if miou > best_miou:
                    best_miou = miou
                    patience_counter = 0
                    save_checkpoint(model, optimizer, scheduler, scaler,
                                    epoch, global_step, best_miou,
                                    output_dir / 'best_model.pth')
                    try:
                        !cp {output_dir}/best_model.pth {DRIVE_OUTPUT}/best_model.pth
                    except Exception:
                        pass
                    print(f'  ✅ New best mIoU={best_miou:.4f} (F1 here={f1:.4f}) — saved to Drive')
                else:
                    patience_counter += 1
                    if patience_counter >= patience:
                        print(f'\n⏹  Early stopping: mIoU did not improve for {patience} evals. '
                              f'Best mIoU={best_miou:.4f}  Best F1={best_f1:.4f}')
                        stop_training = True
                        break

            # ── Periodic checkpoint ───────────────────────────────────────────
            if global_step % save_every == 0 and global_step > 0:
                ckpt_path = output_dir / f'checkpoint_step_{global_step}.pth'
                save_checkpoint(model, optimizer, scheduler, scaler,
                                epoch, global_step, best_miou, ckpt_path)
                try:
                    !cp {ckpt_path} {DRIVE_OUTPUT}/
                except Exception:
                    pass
                cleanup_old_checkpoints(output_dir, keep_last=cfg.train.keep_last_n)

    # ── End of epoch summary ──────────────────────────────────────────────────
    if not stop_training:
        train_summary = epoch_metrics.compute()
        epoch_f1 = train_summary.get('weighted_f1', 0.0)
        if epoch_f1 != epoch_f1: epoch_f1 = 0.0
        print(f'\nEpoch {epoch+1} | '
              f'Loss={train_summary["mean_loss"]:.4f}  '
              f'mIoU={train_summary["mean_iou"]:.4f}  '
              f'F1={epoch_f1:.4f}  '
              f'NaN-skips={nan_skip_count}')
        train_history['loss'].append(train_summary['mean_loss'])
        train_history['miou'].append(train_summary['mean_iou'])
        train_history['f1'].append(epoch_f1)
        train_history['step'].append(global_step)

        if wandb_run:
            wandb_run.log({'train/loss': train_summary['mean_loss'],
                           'train/miou': train_summary['mean_iou'],
                           'train/f1':   epoch_f1,
                           'epoch': epoch+1})

print(f'\nTraining complete!  Best val mIoU = {best_miou:.4f}  |  Best val F1 = {best_f1:.4f}  '
      f'(NaN-skipped batches: {nan_skip_count})')
if wandb_run:
    wandb_run.finish()

## 13. 📈 Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Training Progress', fontsize=14, fontweight='bold')

# Loss
axes[0].plot(range(1, len(train_history['loss'])+1), train_history['loss'],
             color='#E53935', linewidth=2, label='Train Loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

# mIoU
axes[1].plot(range(1, len(train_history['miou'])+1), train_history['miou'],
             color='#1E88E5', linewidth=2, label='Train mIoU')
if val_history['miou']:
    val_steps = [s / (len(train_loader) // grad_accum) for s in val_history['step']]
    axes[1].plot(val_steps, val_history['miou'],
                 color='#43A047', linewidth=2, linestyle='--', label='Val mIoU', marker='o')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('mIoU')
axes[1].set_title('Mean IoU'); axes[1].legend(); axes[1].grid(alpha=0.3)

# F1
axes[2].plot(range(1, len(train_history['f1'])+1), train_history['f1'],
             color='#8E24AA', linewidth=2, label='Train F1')
if val_history['f1']:
    axes[2].plot(val_steps, val_history['f1'],
                 color='#FB8C00', linewidth=2, linestyle='--', label='Val F1', marker='o')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Weighted F1')
axes[2].set_title('Weighted F1'); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
curve_path = f'{OUTPUT_DIR}/training_curves.png'
plt.savefig(curve_path, dpi=120, bbox_inches='tight')
!cp {curve_path} {DRIVE_OUTPUT}/training_curves.png
plt.show()
print('✅ Training curves saved')

## 14. 📐 Evaluate on Test Set

In [ ]:
import subprocess

best_ckpt = f'{OUTPUT_DIR}/best_model.pth'
eval_dir  = f'{OUTPUT_DIR}/eval'

cmd = [
    'python', 'evaluate.py',
    '--checkpoint',           best_ckpt,
    '--data_root',            DATA_ROOT,
    '--split',                'test',
    '--output',               eval_dir,
    '--save_visualizations',
]

print('Running evaluation on test set...')
result = subprocess.run(cmd, capture_output=True, text=True, cwd='/content/sea_ice_seg')
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

# Copy eval results to Drive
!cp -r {eval_dir} {DRIVE_OUTPUT}/
print(f'✅ Evaluation results copied to Drive: {DRIVE_OUTPUT}/eval')

## 15. 🔭 Run Inference on Sample Images

In [ ]:
import sys, torch, shutil
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from PIL import Image
sys.path.insert(0, '/content/sea_ice_seg')

from config import cfg, IDX_TO_ICE_CLASS
from models.pipeline import SeaIceSegmentationPipeline
from data.preprocessing import SARPreprocessor
from data.dataset import binarize_mask

# ── Robust globals (runtime may have been restarted since cells 5/9 ran) ──────
DATA_ROOT  = globals().get('DATA_ROOT',  '/content/sea_ice_seg/dataset')
OUTPUT_DIR = globals().get('OUTPUT_DIR', '/content/sea_ice_seg/outputs')
device     = globals().get('device', 'cuda' if torch.cuda.is_available() else 'cpu')

best_ckpt = f'{OUTPUT_DIR}/best_model.pth'
if not Path(best_ckpt).exists():
    best_ckpt = '/content/drive/MyDrive/sea_ice_seg/outputs/best_model.pth'

if not Path(best_ckpt).exists():
    print(f'⚠️  Checkpoint not found at {best_ckpt}. Run training first!')
else:
    ckpt  = torch.load(best_ckpt, map_location=device)
    state = ckpt['model_state_dict']

    # ── Rebuild the EXACT architecture the checkpoint was trained with ────────
    det_base = state['mask_decoder.enc1.0.weight'].shape[0]
    cfg.model.llm_backend           = 'cross_attn_only'
    cfg.model.decoder_type          = 'unet'
    cfg.model.decoder_base_channels = det_base
    cfg.data.image_size             = (512, 512)

    print(f'Rebuilding model → decoder=unet  base={det_base}  '
          f'llm=cross_attn_only  SAM=False')
    infer_model = SeaIceSegmentationPipeline(cfg.model, use_sam=False).to(device)

    missing, unexpected = infer_model.load_state_dict(state, strict=False)
    critical = [k for k in missing
                if any(s in k for s in ('mask_decoder', 'classifier',
                                        'text_encoder', 'img_proj', 'temporal'))]
    if critical:
        raise RuntimeError(f'Critical trained weights missing: {critical[:8]}')
    infer_model.eval()
    print(f'✅ Loaded best model (val mIoU={ckpt.get("best_metric", float("nan")):.4f})  '
          f'| {len(missing)} missing / {len(unexpected)} unexpected = frozen buffers, OK')

    preprocessor = SARPreprocessor(cfg.data)

    ice_classes = ['Young Ice', 'First Year Ice', 'Floating Ice',
                   'Glaciers', 'Icebergs', 'Old Ice']
    data_root = Path(DATA_ROOT)

    # ── For each class scan MAX_TRIES images and keep the most STRUCTURED one ─
    # A useful demo mask is neither all-background (blank) nor all-foreground
    # (saturated — looked black due to matplotlib auto-scaling). The structure
    # score min(fg, 1-fg) is 0 at both extremes and peaks for a balanced mask,
    # so it rejects the 100%-fg "Floating Ice" case as well as blank masks.
    MAX_TRIES     = 20            # images to scan per class
    STRUCTURE_MIN = 0.10          # min(fg,1-fg) ≥ this → mask has real structure

    fig, axes = plt.subplots(3, 6, figsize=(24, 12))
    fig.suptitle('Inference: Input | GT (Otsu) Mask | Predicted Mask',
                 fontsize=13, fontweight='bold')

    for col, cls in enumerate(ice_classes):
        img_dir  = data_root / cls / 'images'
        mask_dir = data_root / cls / 'masks'
        imgs = sorted(img_dir.glob('*.jpg'))
        if not imgs:
            for ax in axes[:, col]:
                ax.axis('off')
            continue

        best_img_pil   = None
        best_pred_mask = None
        best_pred_name = None
        best_gt_mask   = None
        best_struct    = -1.0
        best_fg        = 0.0

        for img_path in imgs[:MAX_TRIES]:
            stem      = img_path.stem.rstrip('_')
            mask_path = mask_dir / (stem + '_scat.jpg')

            img_pil = Image.open(img_path).convert('L')
            img_np  = np.array(img_pil).astype(np.float32) / 255.0
            img_tensor = preprocessor(img_np).unsqueeze(0).to(device)

            desc = 'Segment the most salient ice region in this SAR image.'
            with torch.no_grad():
                outputs = infer_model(
                    images=img_tensor,
                    descriptions=[desc],
                    sequence_ids=[cls.replace(' ', '_').lower()],
                )

            pred_mask = (outputs['masks'][0, 0] > 0.5).cpu().numpy()
            pred_cls  = int(outputs['pred_class_idx'][0].item())
            pred_name = IDX_TO_ICE_CLASS.get(pred_cls, str(pred_cls))
            fg        = float(pred_mask.mean())
            struct    = min(fg, 1.0 - fg)        # 0 at all-bg or all-fg; peaks at 0.5

            gt_mask_arr = None
            if mask_path.exists():
                gt_gray     = np.array(Image.open(mask_path).convert('L'))
                gt_mask_arr = binarize_mask(gt_gray, mode=cfg.data.mask_binarize)

            if struct > best_struct:
                best_struct, best_fg       = struct, fg
                best_img_pil, best_pred_mask = img_pil, pred_mask
                best_pred_name, best_gt_mask = pred_name, gt_mask_arr

            if struct >= STRUCTURE_MIN:
                break   # structured mask found — stop early

        if best_struct < STRUCTURE_MIN:
            print(f'  ⚠️  {cls}: no well-structured mask in {MAX_TRIES} imgs '
                  f'(best struct={best_struct:.3f}, fg={best_fg*100:.0f}%). '
                  f'Showing the closest one.')

        axes[0, col].imshow(best_img_pil, cmap='gray')
        axes[0, col].set_title(f'Input\n{cls}', fontsize=7)
        axes[0, col].axis('off')

        if best_gt_mask is not None:
            axes[1, col].imshow(best_gt_mask, cmap='gray', vmin=0, vmax=1)
        axes[1, col].set_title('GT (Otsu)', fontsize=7)
        axes[1, col].axis('off')

        # vmin/vmax pinned so an all-fg or all-bg mask still renders correctly
        axes[2, col].imshow(best_pred_mask, cmap='gray', vmin=0, vmax=1)
        axes[2, col].set_title(f'Pred: {best_pred_name}\n({best_fg*100:.1f}% fg)',
                               fontsize=7)
        axes[2, col].axis('off')

    plt.tight_layout()
    demo_path = f'{OUTPUT_DIR}/inference_demo.png'
    plt.savefig(demo_path, dpi=120, bbox_inches='tight')
    try:
        shutil.copy(demo_path,
                    '/content/drive/MyDrive/sea_ice_seg/outputs/inference_demo.png')
    except Exception:
        pass
    plt.show()
    print('✅ Inference demo saved:', demo_path)


## 16. 🧠 Reasoning Segmentation Demo

> **How to use:** Change `DEMO_IMAGE_PATH` to any SAR image path in your dataset,
> and edit `DEMO_PROMPT` to describe what you want the model to segment.
> The model will produce a segmentation mask **and** a text response
> in the LISA-style `"Sure, here is 'Seg'"` format.
> Leave `DEMO_IMAGE_PATH = None` to let the cell auto-select a non-blank sample.

In [ ]:
import sys, torch, shutil
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from PIL import Image as _PILImage
sys.path.insert(0, '/content/sea_ice_seg')

# Reuse globals from previous cells (safe if runtime was not restarted)
DATA_ROOT  = globals().get('DATA_ROOT',  '/content/sea_ice_seg/dataset')
OUTPUT_DIR = globals().get('OUTPUT_DIR', '/content/sea_ice_seg/outputs')
device     = globals().get('device', 'cuda' if torch.cuda.is_available() else 'cpu')

# ════════════════════════════════════════════════════════════════════════════
# ▶  USER-CONFIGURABLE INPUTS  — edit these two lines
# ════════════════════════════════════════════════════════════════════════════
DEMO_IMAGE_PATH = None   # e.g. '/content/sea_ice_seg/dataset/Glaciers/images/abc.jpg'
                         # or None → auto-select a non-blank sample
DEMO_PROMPT = "Segment the sea ice region in this SAR image."
# ════════════════════════════════════════════════════════════════════════════

# Make sure infer_model / preprocessor / ice_classes exist (load if missing)
if 'infer_model' not in dir() or infer_model is None:
    from config import cfg, IDX_TO_ICE_CLASS
    from models.pipeline import SeaIceSegmentationPipeline
    from data.preprocessing import SARPreprocessor
    from data.dataset import binarize_mask

    best_ckpt = f'{OUTPUT_DIR}/best_model.pth'
    if not Path(best_ckpt).exists():
        best_ckpt = '/content/drive/MyDrive/sea_ice_seg/outputs/best_model.pth'

    ckpt  = torch.load(best_ckpt, map_location=device)
    state = ckpt['model_state_dict']
    det_base = state['mask_decoder.enc1.0.weight'].shape[0]
    cfg.model.llm_backend           = 'cross_attn_only'
    cfg.model.decoder_type          = 'unet'
    cfg.model.decoder_base_channels = det_base
    cfg.data.image_size             = (512, 512)

    infer_model  = SeaIceSegmentationPipeline(cfg.model, use_sam=False).to(device)
    missing, _   = infer_model.load_state_dict(state, strict=False)
    infer_model.eval()
    preprocessor = SARPreprocessor(cfg.data)
    ice_classes  = ['Young Ice', 'First Year Ice', 'Floating Ice',
                    'Glaciers', 'Icebergs', 'Old Ice']
    from config import IDX_TO_ICE_CLASS
    print('Model reloaded.')
else:
    from config import cfg, IDX_TO_ICE_CLASS
    from data.preprocessing import SARPreprocessor
    from data.dataset import binarize_mask
    if 'preprocessor' not in dir() or preprocessor is None:
        preprocessor = SARPreprocessor(cfg.data)
    ice_classes = ['Young Ice', 'First Year Ice', 'Floating Ice',
                   'Glaciers', 'Icebergs', 'Old Ice']

# ── Per-class narrative descriptions (honest SAR-based) ──────────────────────
CLASS_DESCRIPTIONS = {
    'Young Ice':      'thin, newly formed sea ice with high microwave transparency '
                      'and characteristically low SAR backscatter',
    'First Year Ice': 'seasonal ice formed within a single winter, exhibiting '
                      'moderate and relatively uniform SAR backscatter',
    'Floating Ice':   'fragmented sea ice drifting on the ocean surface, appearing '
                      'as irregular bright patches against darker open water',
    'Glaciers':       'permanent land-based ice masses with smooth surface texture '
                      'and characteristically low SAR backscatter zones',
    'Icebergs':       'large floating ice blocks calved from glaciers, appearing '
                      'as bright isolated high-backscatter targets in SAR imagery',
    'Old Ice':        'multi-year sea ice with a rough, deformed surface caused by '
                      'repeated melt/refreeze cycles, producing high SAR backscatter',
}

data_root = Path(DATA_ROOT)

# ── Auto-pick an image if none specified ──────────────────────────────────────
if DEMO_IMAGE_PATH is None:
    print('Auto-searching for a non-blank sample (foreground > 5%)…')
    _found = False
    for _cls in ice_classes:
        _img_dir = data_root / _cls / 'images'
        for _ip in sorted(_img_dir.glob('*.jpg'))[:15]:
            _pil = _PILImage.open(_ip).convert('L')
            _np  = np.array(_pil).astype(np.float32) / 255.0
            _t   = preprocessor(_np).unsqueeze(0).to(device)
            with torch.no_grad():
                _o = infer_model(images=_t, descriptions=[DEMO_PROMPT],
                                 sequence_ids=[_cls.replace(' ', '_').lower()])
            _fg = (_o['masks'][0, 0] > 0.5).float().mean().item()
            if _fg > 0.05:
                DEMO_IMAGE_PATH = str(_ip)
                print(f'  Selected: {_cls}/{_ip.name}  (fg={_fg*100:.1f}%)')
                _found = True
                break
        if _found:
            break
    if not _found:
        # Absolute fallback: first image in dataset
        for _cls in ice_classes:
            _imgs = sorted((data_root / _cls / 'images').glob('*.jpg'))
            if _imgs:
                DEMO_IMAGE_PATH = str(_imgs[0])
                print(f'  Fallback: {DEMO_IMAGE_PATH}')
                break

# ── Identify class from folder name ──────────────────────────────────────────
demo_img_path = Path(DEMO_IMAGE_PATH)
demo_cls      = demo_img_path.parent.parent.name   # folder = ice class name

# ── Run reasoning segmentation inference ─────────────────────────────────────
img_pil    = _PILImage.open(demo_img_path).convert('L')
img_np     = np.array(img_pil).astype(np.float32) / 255.0
img_tensor = preprocessor(img_np).unsqueeze(0).to(device)

with torch.no_grad():
    demo_out = infer_model(
        images=img_tensor,
        descriptions=[DEMO_PROMPT],
        sequence_ids=[demo_cls.replace(' ', '_').lower()],
    )

pred_mask_prob = demo_out['masks'][0, 0].cpu().numpy()        # (H, W) float
pred_mask_bin  = (pred_mask_prob > 0.5).astype(np.uint8)
pred_cls_idx   = int(demo_out['pred_class_idx'][0].item())
pred_cls_name  = IDX_TO_ICE_CLASS.get(pred_cls_idx, str(pred_cls_idx))
fg_pct         = pred_mask_bin.mean() * 100

# ── Build LISA-style text response ───────────────────────────────────────────
ice_desc = CLASS_DESCRIPTIONS.get(
    pred_cls_name, 'ice with characteristic SAR backscatter patterns')
text_response = (
    f"Sure, here is the segmentation of {pred_cls_name}: [SEG]\n"
    f"\n"
    f"The model identified this SAR scene as {pred_cls_name} — {ice_desc}.\n"
    f"{fg_pct:.1f}% of image pixels were classified as ice foreground."
)

# ── Build red-tinted mask overlay ────────────────────────────────────────────
img_small = np.array(
    _PILImage.fromarray((img_np * 255).astype(np.uint8)).resize(
        (pred_mask_bin.shape[1], pred_mask_bin.shape[0]), _PILImage.BILINEAR))
overlay_rgb = np.stack([img_small] * 3, axis=-1).astype(np.float32)
red_channel = np.array([255, 60, 60], dtype=np.float32)
mask_3d     = pred_mask_bin[:, :, np.newaxis]
overlay_rgb = (overlay_rgb * (1 - 0.55 * mask_3d)
               + red_channel * 0.55 * mask_3d).clip(0, 255).astype(np.uint8)

# ── Visualise ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
short_prompt = DEMO_PROMPT if len(DEMO_PROMPT) < 70 else DEMO_PROMPT[:67] + '…'
fig.suptitle(
    f'Reasoning Segmentation Demo\nPrompt: "{short_prompt}"',
    fontsize=11, fontweight='bold')

axes[0].imshow(img_pil, cmap='gray')
axes[0].set_title(f'Input SAR Image\n({demo_cls})', fontsize=10)
axes[0].axis('off')

axes[1].imshow(overlay_rgb)
axes[1].set_title(f'Predicted Mask Overlay\nModel: {pred_cls_name}  ({fg_pct:.1f}% fg)',
                  fontsize=10)
axes[1].axis('off')

im = axes[2].imshow(pred_mask_prob, cmap='hot', vmin=0, vmax=1)
axes[2].set_title('Segmentation Probability\nHeatmap', fontsize=10)
axes[2].axis('off')
plt.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

plt.tight_layout()
seg_path = f'{OUTPUT_DIR}/reasoning_seg_demo.png'
plt.savefig(seg_path, dpi=120, bbox_inches='tight')
try:
    shutil.copy(seg_path,
                '/content/drive/MyDrive/sea_ice_seg/outputs/reasoning_seg_demo.png')
except Exception:
    pass
plt.show()

# ── Print the text response ───────────────────────────────────────────────────
print('\n' + '─' * 62)
print('📝  Model Reasoning Response:')
print('─' * 62)
print(text_response)
print('─' * 62)
print(f'\n✅  Demo saved: {seg_path}')


## 17. 📏 Full Metric Suite (cIoU, gIoU, Boundary IoU, BLEU, ROUGE-L, CIDEr)

Computes the complete set of paper metrics on the **test set**:

**Segmentation** — mIoU, gIoU (mean per-image IoU), cIoU (cumulative IoU), Boundary IoU, pixel Precision/Recall, Dice, pixel accuracy.

**Classification** — accuracy, macro-F1, weighted-F1, per-class F1.

**Reasoning text** — BLEU-1..4, ROUGE-L, CIDEr.

> ⚠️ **Honest note on the text metrics:** the trained model uses `llm_backend='cross_attn_only'`, which does **not** free-generate text — it classifies the ice type and the reasoning sentence is *templated* from the predicted class. BLEU/ROUGE/CIDEr here therefore measure how well the predicted-class description matches the ground-truth annotation (a text-space reflection of classification quality), **not** free caption generation. Report them with this framing.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  FULL METRIC SUITE — segmentation (cIoU/gIoU/BoundaryIoU/P/R) + text     ║
# ║  (BLEU-1..4, ROUGE-L, CIDEr) on the test set                            ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import sys, torch, json, shutil, subprocess
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm
sys.path.insert(0, '/content/sea_ice_seg')

# Pull latest metrics code (utils/metrics.py + utils/text_metrics.py)
subprocess.run(['git', '-C', '/content/sea_ice_seg', 'fetch', 'origin',
                'claude/laughing-thompson-AhCX7'], capture_output=True)
subprocess.run(['git', '-C', '/content/sea_ice_seg', 'checkout',
                'origin/claude/laughing-thompson-AhCX7', '--',
                'utils/metrics.py', 'utils/text_metrics.py'], capture_output=True)
import importlib, utils.metrics as _m, utils.text_metrics as _tm
importlib.reload(_m); importlib.reload(_tm)
print('✅ metrics + text_metrics reloaded')

DATA_ROOT  = globals().get('DATA_ROOT',  '/content/sea_ice_seg/dataset')
OUTPUT_DIR = globals().get('OUTPUT_DIR', '/content/sea_ice_seg/outputs')
device     = globals().get('device', 'cuda' if torch.cuda.is_available() else 'cpu')

from config import cfg, ICE_CLASSES, IDX_TO_ICE_CLASS
from models.pipeline import SeaIceSegmentationPipeline
from data.dataset import build_dataloaders
from utils.metrics import MetricAccumulator
from utils.text_metrics import compute_all_text_metrics

# ── Per-class canonical descriptions (the templated reasoning text source) ───
# These mirror the demo's CLASS_DESCRIPTIONS — honest, SAR-grounded sentences.
CLASS_DESC = {
    'Young Ice':      'thin newly formed sea ice with high microwave transparency and low SAR backscatter',
    'First Year Ice': 'seasonal ice formed within a single winter with moderate uniform SAR backscatter',
    'Floating Ice':   'fragmented sea ice drifting on the ocean surface as irregular bright patches',
    'Glaciers':       'permanent land based ice masses with smooth surface texture and low SAR backscatter',
    'Icebergs':       'large floating ice blocks calved from glaciers as bright isolated high backscatter targets',
    'Old Ice':        'multi year sea ice with a rough deformed surface and high SAR backscatter',
}
def _response_text(cls_name):
    return f"sure here is the segmentation of {cls_name} : {CLASS_DESC.get(cls_name, 'sea ice region')}"

# ── Load best checkpoint ──────────────────────────────────────────────────────
best_ckpt = f'{OUTPUT_DIR}/best_model.pth'
if not Path(best_ckpt).exists():
    best_ckpt = '/content/drive/MyDrive/sea_ice_seg/outputs/best_model.pth'
ckpt  = torch.load(best_ckpt, map_location=device)
state = ckpt['model_state_dict']
det_base = state['mask_decoder.enc1.0.weight'].shape[0]
cfg.model.llm_backend           = 'cross_attn_only'
cfg.model.decoder_type          = 'unet'
cfg.model.decoder_base_channels = det_base
cfg.data.data_root              = DATA_ROOT
cfg.data.image_size             = (512, 512)

met_model = SeaIceSegmentationPipeline(cfg.model, use_sam=False).to(device)
met_model.load_state_dict(state, strict=False)
met_model.eval()
print(f'✅ Model loaded  val mIoU={ckpt.get("best_metric", float("nan")):.4f}')

# ── Test loader ───────────────────────────────────────────────────────────────
if 'test_loader' in dir() and test_loader is not None:
    _tl = test_loader
else:
    _, _, _tl = build_dataloaders(cfg.data, cfg.train)
print(f'Test loader: {len(_tl)} batches')

# ── Run evaluation ────────────────────────────────────────────────────────────
acc = MetricAccumulator()
hyps, refs = [], []
import uuid as _uuid
_uid = _uuid.uuid4().hex[:8]

with torch.no_grad():
    for bidx, batch in enumerate(tqdm(_tl, desc='Full-metric eval', leave=False)):
        images = batch['image'].to(device)
        masks  = batch['mask'].to(device)
        labels = batch['label'].to(device)
        descs  = batch['long_desc']
        seqs   = [f'metric_{_uid}_' + str(Path(p).parent.parent.name)
                  for p in batch['image_path']]

        outputs = met_model(images=images, descriptions=descs, sequence_ids=seqs)
        acc.update(outputs=outputs,
                   targets={'mask': masks, 'label': labels},
                   loss=0.0)

        # ── Reasoning-text hyp/ref pairs (see honest note above) ─────────────
        for i in range(images.shape[0]):
            pred_name = IDX_TO_ICE_CLASS.get(int(outputs['pred_class_idx'][i].item()), '')
            hyps.append(_response_text(pred_name))
            # reference = the dataset's own long description for this sample
            refs.append(str(descs[i]))

# Clean temporal bank
for k in list(met_model.temporal._banks.keys()):
    if k.startswith(f'metric_{_uid}'): del met_model.temporal._banks[k]

seg = acc.compute()
txt = compute_all_text_metrics(hyps, refs)

# ── Assemble & save ───────────────────────────────────────────────────────────
full_metrics = {
    'segmentation': {
        'mIoU':         round(seg['mean_iou'], 4),
        'gIoU':         round(seg['giou'], 4),
        'cIoU':         round(seg['ciou'], 4),
        'Boundary_IoU': round(seg['boundary_iou'], 4),
        'Precision':    round(seg['seg_precision'], 4),
        'Recall':       round(seg['seg_recall'], 4),
        'Dice':         round(seg['mean_dice'], 4),
        'Pixel_Acc':    round(seg['pixel_accuracy'], 4),
    },
    'classification': {
        'Accuracy':    round(seg['accuracy'], 4),
        'Macro_F1':    round(seg['macro_f1'], 4),
        'Weighted_F1': round(seg['weighted_f1'], 4),
        'Per_Class_F1': {k: round(v, 4) for k, v in seg['per_class_f1'].items()},
    },
    'reasoning_text': txt,
    'text_metrics_note': ('cross_attn_only backend: reasoning text is templated '
                          'from the predicted class, so BLEU/ROUGE/CIDEr reflect '
                          'class-description alignment, not free generation.'),
}

out_json = f'{OUTPUT_DIR}/full_metrics.json'
with open(out_json, 'w') as f: json.dump(full_metrics, f, indent=2)
try: shutil.copy(out_json, '/content/drive/MyDrive/sea_ice_seg/outputs/full_metrics.json')
except Exception: pass

# ── Pretty print ──────────────────────────────────────────────────────────────
def _section(title, d):
    print(f'\n{title}')
    print('─' * 46)
    for k, v in d.items():
        if isinstance(v, dict):
            print(f'  {k}:')
            for kk, vv in v.items():
                print(f'    {kk:<22} {vv:.4f}')
        else:
            print(f'  {k:<24} {v:.4f}')

_section('SEGMENTATION METRICS', full_metrics['segmentation'])
_section('CLASSIFICATION METRICS', full_metrics['classification'])
_section('REASONING-TEXT METRICS', full_metrics['reasoning_text'])
print('\n⚠️  ' + full_metrics['text_metrics_note'])

# ── LaTeX block ───────────────────────────────────────────────────────────────
s, c, t = (full_metrics['segmentation'], full_metrics['classification'],
           full_metrics['reasoning_text'])
print('\n' + '─' * 60)
print('LaTeX (copy into paper):')
print('─' * 60)
print(r'\begin{table}[t]\centering')
print(r'\caption{Full evaluation of the proposed sea ice reasoning '
      r'segmentation model on the held-out test set.}')
print(r'\label{tab:full_metrics}')
print(r'\begin{tabular}{lc}')
print(r'\toprule')
print(r'Metric & Value \\')
print(r'\midrule')
print(r'\multicolumn{2}{l}{\textit{Segmentation}} \\')
print(rf"\quad mIoU / gIoU & {s['mIoU']:.4f} \\")
print(rf"\quad cIoU & {s['cIoU']:.4f} \\")
print(rf"\quad Boundary IoU & {s['Boundary_IoU']:.4f} \\")
print(rf"\quad Precision / Recall & {s['Precision']:.4f} / {s['Recall']:.4f} \\")
print(rf"\quad Dice & {s['Dice']:.4f} \\")
print(r'\multicolumn{2}{l}{\textit{Classification}} \\')
print(rf"\quad Weighted F1 & {c['Weighted_F1']:.4f} \\")
print(rf"\quad Macro F1 & {c['Macro_F1']:.4f} \\")
print(r'\multicolumn{2}{l}{\textit{Reasoning text}} \\')
print(rf"\quad BLEU-4 & {t['BLEU-4']:.4f} \\")
print(rf"\quad ROUGE-L & {t['ROUGE-L']:.4f} \\")
print(rf"\quad CIDEr & {t['CIDEr']:.4f} \\")
print(r'\bottomrule')
print(r'\end{tabular}')
print(r'\end{table}')
print(f'\n✅ Saved → {out_json}')


## 18. 🔬 Ablation Studies

Three cells cover all standard paper ablations:
- **Cell A** (~10 min): inference-time knockouts — no retraining needed
- **Cell B** (~15–20 min / variant): architecture & loss retraining ablations
- **Cell C** (instant): print consolidated results table + LaTeX source

All results are accumulated in `outputs/ablation_results.json` so cells can be re-run independently.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  ABLATION — Cell A: Fast inference-time ablations (no retraining)       ║
# ║  Variants: Full | w/o Temporal | w/o Depth | Threshold sweep            ║
# ║  Reports full suite: mIoU/gIoU, cIoU, Boundary IoU, P/R, Dice, F1       ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import sys, torch, json, shutil, uuid, subprocess
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm
sys.path.insert(0, '/content/sea_ice_seg')

# Pull latest metrics code
subprocess.run(['git', '-C', '/content/sea_ice_seg', 'fetch', 'origin',
                'claude/laughing-thompson-AhCX7'], capture_output=True)
subprocess.run(['git', '-C', '/content/sea_ice_seg', 'checkout',
                'origin/claude/laughing-thompson-AhCX7', '--',
                'utils/metrics.py'], capture_output=True)
import importlib, utils.metrics as _m
importlib.reload(_m)

DATA_ROOT  = globals().get('DATA_ROOT',  '/content/sea_ice_seg/dataset')
OUTPUT_DIR = globals().get('OUTPUT_DIR', '/content/sea_ice_seg/outputs')
device     = globals().get('device', 'cuda' if torch.cuda.is_available() else 'cpu')
ABL_JSON   = f'{OUTPUT_DIR}/ablation_results.json'

from config import cfg, IDX_TO_ICE_CLASS
from models.pipeline import SeaIceSegmentationPipeline
from data.dataset import build_dataloaders
from utils.losses import SeaIceLoss
from utils.metrics import (compute_iou, compute_dice, compute_pixel_accuracy,
                           compute_giou, compute_boundary_iou,
                           compute_precision_recall, compute_intersection_union,
                           ClassificationMetrics)

# ── Load best checkpoint ──────────────────────────────────────────────────────
best_ckpt = f'{OUTPUT_DIR}/best_model.pth'
if not Path(best_ckpt).exists():
    best_ckpt = '/content/drive/MyDrive/sea_ice_seg/outputs/best_model.pth'
ckpt  = torch.load(best_ckpt, map_location=device)
state = ckpt['model_state_dict']
det_base = state['mask_decoder.enc1.0.weight'].shape[0]
cfg.model.llm_backend           = 'cross_attn_only'
cfg.model.decoder_type          = 'unet'
cfg.model.decoder_base_channels = det_base
cfg.data.data_root              = DATA_ROOT
cfg.data.image_size             = (512, 512)

abl_model = SeaIceSegmentationPipeline(cfg.model, use_sam=False).to(device)
abl_model.load_state_dict(state, strict=False)
abl_model.eval()
print(f'✅ Model loaded  val mIoU={ckpt.get("best_metric", float("nan")):.4f}')

if 'test_loader' in dir() and test_loader is not None:
    _test_loader = test_loader
else:
    _, _, _test_loader = build_dataloaders(cfg.data, cfg.train)
    print(f'Built test loader: {len(_test_loader)} batches')

abl_criterion = SeaIceLoss(cfg.train).to(device)

# ── Core eval helper (full suite, threshold-parametrised) ─────────────────────
def _abl_eval(model, loader, criterion, device,
              no_temporal=False, depth_zero=False, threshold=0.5):
    model.eval()
    hook = None
    if depth_zero:
        hook = model.depth_encoder.register_forward_hook(
            lambda m, i, o: torch.zeros_like(o))

    ious, dices, pxs, bious, precs, recs, losses = [], [], [], [], [], [], []
    cum_i = cum_u = 0.0
    cls_m = ClassificationMetrics()
    _uid  = uuid.uuid4().hex[:8]

    with torch.no_grad():
        for bidx, batch in enumerate(loader):
            images = batch['image'].to(device)
            masks  = batch['mask'].to(device)
            labels = batch['label'].to(device)
            descs  = batch['long_desc']
            if no_temporal:
                seqs = [f'abl_{_uid}_{bidx}_{i}' for i in range(images.shape[0])]
            else:
                seqs = [f'abl_{_uid}_' + str(Path(p).parent.parent.name)
                        for p in batch['image_path']]

            outputs = model(images=images, descriptions=descs, sequence_ids=seqs)
            loss_d  = criterion(outputs, {'mask': masks, 'label': labels})
            mk = outputs['masks']

            ious.append( compute_iou(mk, masks, threshold=threshold))
            dices.append(compute_dice(mk, masks, threshold=threshold))
            pxs.append(  compute_pixel_accuracy(mk, masks, threshold=threshold))
            bious.append(compute_boundary_iou(mk, masks, threshold=threshold))
            _p, _r = compute_precision_recall(mk, masks, threshold=threshold)
            precs.append(_p); recs.append(_r)
            _i, _u = compute_intersection_union(mk, masks, threshold=threshold)
            cum_i += _i; cum_u += _u
            losses.append(loss_d['loss'].item())
            cls_m.update(outputs['pred_class_idx'], labels)

    for k in list(model.temporal._banks.keys()):
        if k.startswith(f'abl_{_uid}'): del model.temporal._banks[k]
    if hook: hook.remove()

    cr = cls_m.compute()
    return {
        'mean_iou':       round(float(np.mean(ious)),   4),
        'giou':           round(float(np.mean(ious)),   4),   # mean per-image IoU
        'ciou':           round(float(cum_i / cum_u) if cum_u > 0 else 0.0, 4),
        'boundary_iou':   round(float(np.mean(bious)),  4),
        'seg_precision':  round(float(np.mean(precs)),  4),
        'seg_recall':     round(float(np.mean(recs)),   4),
        'mean_dice':      round(float(np.mean(dices)),  4),
        'pixel_accuracy': round(float(np.mean(pxs)),    4),
        'mean_loss':      round(float(np.mean(losses)), 4),
        'weighted_f1':    round(float(cr['weighted_f1']), 4),
        'macro_f1':       round(float(cr['macro_f1']),    4),
    }

# ── Run variants ──────────────────────────────────────────────────────────────
abl_results = {}
if Path(ABL_JSON).exists():
    with open(ABL_JSON) as f: abl_results = json.load(f)

variants_A = [
    ('Full Model (Ours)',         dict()),
    ('w/o Temporal Consistency',  dict(no_temporal=True)),
    ('w/o Depth Features',        dict(depth_zero=True)),
]
for name, kwargs in variants_A:
    print(f'Running: {name}...', end=' ', flush=True)
    abl_results[name] = _abl_eval(abl_model, _test_loader, abl_criterion, device, **kwargs)
    r = abl_results[name]
    print(f'mIoU={r["mean_iou"]:.4f}  cIoU={r["ciou"]:.4f}  '
          f'BIoU={r["boundary_iou"]:.4f}  F1={r["weighted_f1"]:.4f}')

print('\nThreshold sensitivity sweep:')
for thresh in [0.3, 0.4, 0.5, 0.6, 0.7]:
    key = f'Threshold={thresh}'
    abl_results[key] = _abl_eval(abl_model, _test_loader, abl_criterion,
                                 device, threshold=thresh)
    r = abl_results[key]
    marker = '  ← default' if thresh == 0.5 else ''
    print(f'  thresh={thresh}  mIoU={r["mean_iou"]:.4f}  cIoU={r["ciou"]:.4f}  '
          f'F1={r["weighted_f1"]:.4f}{marker}')

# ── Save & display ────────────────────────────────────────────────────────────
with open(ABL_JSON, 'w') as f: json.dump(abl_results, f, indent=2)
try: shutil.copy(ABL_JSON, '/content/drive/MyDrive/sea_ice_seg/outputs/ablation_results.json')
except Exception: pass

print(f'\n{"Variant":<28}{"mIoU":>8}{"cIoU":>8}{"BIoU":>8}{"Dice":>8}{"F1":>8}')
print('-' * 68)
for name, _ in variants_A:
    r = abl_results[name]
    print(f'{name:<28}{r["mean_iou"]:>8.4f}{r["ciou"]:>8.4f}'
          f'{r["boundary_iou"]:>8.4f}{r["mean_dice"]:>8.4f}{r["weighted_f1"]:>8.4f}')
print(f'\n✅ Saved → {ABL_JSON}')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  ABLATION — Cell B: Architecture & loss retraining ablations            ║
# ║  Each variant trains ≤15 epochs (patience=5) then evaluates test set.   ║
# ║  Time: ~15–20 min per variant on A100  (~2 hrs for all 6)               ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import sys, torch, json, shutil, os
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm
sys.path.insert(0, '/content/sea_ice_seg')

# Pull latest code (gets lora_rank=0 fix in visual_encoder.py)
import subprocess
subprocess.run(['git', '-C', '/content/sea_ice_seg', 'fetch', 'origin',
                'claude/laughing-thompson-AhCX7'], capture_output=True)
subprocess.run(['git', '-C', '/content/sea_ice_seg', 'checkout',
                'origin/claude/laughing-thompson-AhCX7', '--',
                'models/visual_encoder.py'], capture_output=True)
import importlib, models.visual_encoder as _ve
importlib.reload(_ve)
print('✅ visual_encoder.py reloaded (lora_rank=0 support)')

from torch.amp import GradScaler, autocast
from config import Config
from models.pipeline import SeaIceSegmentationPipeline
from data.dataset import build_dataloaders
from utils.losses import SeaIceLoss, MaskLoss, FocalLoss, TverskyLoss
from utils.metrics import (compute_iou, compute_dice, compute_pixel_accuracy,
                            compute_boundary_iou, compute_precision_recall,
                            compute_intersection_union, ClassificationMetrics)
from train import set_seed, get_lr_scheduler, validate

DATA_ROOT  = globals().get('DATA_ROOT',  '/content/sea_ice_seg/dataset')
OUTPUT_DIR = globals().get('OUTPUT_DIR', '/content/sea_ice_seg/outputs')
device     = globals().get('device', 'cuda' if torch.cuda.is_available() else 'cpu')
ABL_JSON   = f'{OUTPUT_DIR}/ablation_results.json'
FINETUNE   = f'{OUTPUT_DIR}/best_model.pth'  # starting weights (strict=False)
if not Path(FINETUNE).exists():
    FINETUNE = '/content/drive/MyDrive/sea_ice_seg/outputs/best_model.pth'

# ── Base config factory ───────────────────────────────────────────────────────
def _make_cfg(model_overrides=None, train_overrides=None):
    c = Config()
    _ckpt = torch.load(FINETUNE, map_location='cpu')
    _base  = _ckpt['model_state_dict']['mask_decoder.enc1.0.weight'].shape[0]
    del _ckpt
    c.model.llm_backend           = 'cross_attn_only'
    c.model.decoder_type          = 'unet'
    c.model.decoder_base_channels = _base
    c.data.data_root              = DATA_ROOT
    c.data.image_size             = (512, 512)
    c.data.mask_resize_mode       = 'stretch'
    c.train.batch_size            = 4
    c.train.grad_accum_steps      = 4
    c.train.lr                    = 2e-5
    c.train.lora_lr               = 5e-5
    c.train.cls_head_lr           = 5e-5
    c.train.decoder_lr            = 1e-4
    c.train.weight_decay          = 0.05
    c.train.betas                 = (0.9, 0.999)
    c.train.scheduler             = 'cosine'
    c.train.warmup_ratio          = 0.05
    c.train.grad_clip_norm        = 1.0
    c.train.early_stop_patience   = 5
    c.train.num_workers           = 4
    _vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    c.train.fp16 = False if _vram >= 38 else True
    c.train.bf16 = True  if _vram >= 38 else False
    c.train.device = device
    c.train.seed   = 42
    for k, v in (model_overrides or {}).items(): setattr(c.model, k, v)
    for k, v in (train_overrides or {}).items(): setattr(c.train, k, v)
    return c

# ── Compact evaluation helper (no hook version, reuse from Cell A if available)
import uuid
def _eval(model, loader, criterion, dev, desc='Test'):
    model.eval()
    ious, dices, pxs, bious, precs, recs, losses = [], [], [], [], [], [], []
    cum_i = cum_u = 0.0
    cls_m = ClassificationMetrics()
    _uid  = uuid.uuid4().hex[:8]
    with torch.no_grad():
        for bidx, batch in enumerate(tqdm(loader, desc=desc, leave=False)):
            imgs   = batch['image'].to(dev)
            masks  = batch['mask'].to(dev)
            labels = batch['label'].to(dev)
            seqs   = [f'e_{_uid}_'+str(Path(p).parent.parent.name) for p in batch['image_path']]
            out    = model(images=imgs, descriptions=batch['long_desc'], sequence_ids=seqs)
            ld     = criterion(out, {'mask': masks, 'label': labels})
            mk = out['masks']
            ious.append(   compute_iou(mk, masks))
            dices.append(  compute_dice(mk, masks))
            pxs.append(    compute_pixel_accuracy(mk, masks))
            bious.append(  compute_boundary_iou(mk, masks))
            _p, _r = compute_precision_recall(mk, masks)
            precs.append(_p); recs.append(_r)
            _i, _u = compute_intersection_union(mk, masks)
            cum_i += _i; cum_u += _u
            losses.append( ld['loss'].item())
            cls_m.update(out['pred_class_idx'], labels)
    for k in list(model.temporal._banks.keys()):
        if k.startswith(f'e_{_uid}'): del model.temporal._banks[k]
    cr = cls_m.compute()
    return {'mean_iou': round(float(np.mean(ious)),4),
            'giou': round(float(np.mean(ious)),4),
            'ciou': round(float(cum_i / cum_u) if cum_u > 0 else 0.0, 4),
            'boundary_iou': round(float(np.mean(bious)),4),
            'seg_precision': round(float(np.mean(precs)),4),
            'seg_recall': round(float(np.mean(recs)),4),
            'mean_dice': round(float(np.mean(dices)),4),
            'pixel_accuracy': round(float(np.mean(pxs)),4),
            'mean_loss': round(float(np.mean(losses)),4),
            'weighted_f1': round(float(cr['weighted_f1']),4),
            'macro_f1': round(float(cr['macro_f1']),4)}

# ── Compact short-training function ──────────────────────────────────────────
def _short_train(v_name, v_cfg, v_crit, v_tr, v_va, v_te, v_dev,
                 finetune_ckpt=FINETUNE, epochs=15, patience=5):
    print(f'\n[ABL] {v_name}: building model...')
    use_sam = False
    m = SeaIceSegmentationPipeline(v_cfg.model, use_sam=use_sam).to(v_dev)

    # Load matching weights (strict=False: new decoder/arch parts start random)
    if Path(finetune_ckpt).exists():
        _s = torch.load(finetune_ckpt, map_location=v_dev)['model_state_dict']
        miss, _ = m.load_state_dict(_s, strict=False)
        print(f'  Weights loaded  ({len(miss)} missing → randomly init)')
        del _s

    set_seed(v_cfg.train.seed)
    param_groups = m.get_param_groups(v_cfg.model, v_cfg.train)
    optim = torch.optim.AdamW(param_groups, lr=v_cfg.train.lr,
                              weight_decay=v_cfg.train.weight_decay, betas=v_cfg.train.betas)
    spe   = len(v_tr) // v_cfg.train.grad_accum_steps
    sched = get_lr_scheduler(optim, v_cfg.train, spe * epochs)
    use_bf16 = v_cfg.train.bf16
    use_fp16 = v_cfg.train.fp16 and not use_bf16
    scaler   = GradScaler('cuda') if use_fp16 else None
    use_amp  = use_bf16 or use_fp16
    amp_dt   = torch.bfloat16 if use_bf16 else torch.float16
    ga       = v_cfg.train.grad_accum_steps
    cn       = v_cfg.train.grad_clip_norm
    out_dir  = Path(OUTPUT_DIR) / f'abl_{v_name.replace(" ","_").replace("/","")}' 
    out_dir.mkdir(parents=True, exist_ok=True)

    best_miou, pat_cnt = 0.0, 0

    for ep in range(epochs):
        m.train(); m.temporal.reset_all()
        for bidx, batch in enumerate(tqdm(v_tr,
                                          desc=f'  [{v_name}] ep{ep+1}/{epochs}',
                                          leave=False)):
            imgs   = batch['image'].to(v_dev);  msks = batch['mask'].to(v_dev)
            lbs    = batch['label'].to(v_dev);  dsc  = batch['long_desc']
            seqs   = [str(Path(p).parent.parent.name) for p in batch['image_path']]

            with autocast('cuda', dtype=amp_dt, enabled=use_amp):
                out  = m(images=imgs, descriptions=dsc, sequence_ids=seqs, return_aux=True)
                ld   = v_crit(out, {'mask': msks, 'label': lbs})
                loss = ld['loss'] / ga

            if not torch.isfinite(loss):
                optim.zero_grad(); m.temporal.reset_all(); continue

            if scaler: scaler.scale(loss).backward()
            else:      loss.backward()

            if (bidx + 1) % ga == 0:
                if scaler: scaler.unscale_(optim)
                gn = torch.nn.utils.clip_grad_norm_(m.parameters(), cn)
                if torch.isfinite(gn):
                    if scaler: scaler.step(optim)
                    else: optim.step()
                if scaler: scaler.update()
                optim.zero_grad(); sched.step()

        vm = validate(m, v_va, v_crit, v_dev)
        miou = vm['mean_iou']; f1 = vm.get('weighted_f1', 0.0)
        print(f'  [{v_name}] ep{ep+1}: val mIoU={miou:.4f}  F1={f1:.4f}  ',
              f'(best={best_miou:.4f} pat={pat_cnt}/{patience})')
        if miou > best_miou:
            best_miou = miou; pat_cnt = 0
            torch.save({'model_state_dict': m.state_dict(), 'best_metric': best_miou},
                       out_dir / 'best.pth')
        else:
            pat_cnt += 1
            if pat_cnt >= patience:
                print(f'  [{v_name}] Early stop'); break

    if (out_dir / 'best.pth').exists():
        m.load_state_dict(torch.load(out_dir / 'best.pth', map_location=v_dev)['model_state_dict'],
                          strict=False)
    test_m = _eval(m, v_te, v_crit, v_dev, desc=f'Test {v_name}')
    test_m['val_best_miou'] = round(best_miou, 4)
    del m; torch.cuda.empty_cache()
    return test_m

# ── Build shared dataloaders ──────────────────────────────────────────────────
_base_cfg = _make_cfg()
_n_imgs   = len(list(Path(DATA_ROOT).rglob('images/*.jpg')))
_base_cfg.train.eval_every = max(10, _n_imgs // _base_cfg.train.batch_size // _base_cfg.train.grad_accum_steps)
_tr, _va, _te = build_dataloaders(_base_cfg.data, _base_cfg.train)
print(f'Dataloaders ready — train={len(_tr)} val={len(_va)} test={len(_te)} batches')

# ── Load existing results ─────────────────────────────────────────────────────
abl_results = {}
if Path(ABL_JSON).exists():
    with open(ABL_JSON) as f: abl_results = json.load(f)

# ══════════════════════════════════════════════════════════════════════════════
# VARIANT 1: w/o LoRA  (frozen CLIP, no SAR adaptation)
# ══════════════════════════════════════════════════════════════════════════════
v_cfg = _make_cfg(model_overrides={'lora_rank': 0})
v_crit = SeaIceLoss(v_cfg.train).to(device)
abl_results['w/o LoRA (Frozen CLIP)'] = _short_train(
    'w/o LoRA', v_cfg, v_crit, _tr, _va, _te, device,
    finetune_ckpt=None)  # no LoRA → can't load from LoRA checkpoint

# ══════════════════════════════════════════════════════════════════════════════
# VARIANT 2: Token Decoder  (coarse grid decoder, not full-res U-Net)
# ══════════════════════════════════════════════════════════════════════════════
v_cfg = _make_cfg(model_overrides={'decoder_type': 'token'})
v_crit = SeaIceLoss(v_cfg.train).to(device)
abl_results['Token Decoder'] = _short_train(
    'Token Decoder', v_cfg, v_crit, _tr, _va, _te, device)

# ══════════════════════════════════════════════════════════════════════════════
# VARIANT 3: w/o Auxiliary Loss  (remove deep supervision)
# ══════════════════════════════════════════════════════════════════════════════
v_cfg = _make_cfg(train_overrides={'lambda_aux': 0.0})
v_crit = SeaIceLoss(v_cfg.train).to(device)
abl_results['w/o Auxiliary Loss'] = _short_train(
    'w/o Aux Loss', v_cfg, v_crit, _tr, _va, _te, device)

# ══════════════════════════════════════════════════════════════════════════════
# VARIANT 4: Focal Loss Only  (no Tversky/Dice term)
# ══════════════════════════════════════════════════════════════════════════════
v_cfg = _make_cfg()
v_crit = SeaIceLoss(v_cfg.train).to(device)
v_crit.mask_loss.tversky_w = 0.0          # zero out Tversky weight
abl_results['Focal Loss Only'] = _short_train(
    'Focal Only', v_cfg, v_crit, _tr, _va, _te, device)

# ══════════════════════════════════════════════════════════════════════════════
# VARIANT 5: Tversky/Dice Loss Only  (no Focal term)
# ══════════════════════════════════════════════════════════════════════════════
v_cfg = _make_cfg()
v_crit = SeaIceLoss(v_cfg.train).to(device)
v_crit.mask_loss.focal_w = 0.0            # zero out Focal weight
abl_results['Tversky Loss Only'] = _short_train(
    'Tversky Only', v_cfg, v_crit, _tr, _va, _te, device)

# ══════════════════════════════════════════════════════════════════════════════
# VARIANT 6: BCE + Dice  (standard baseline loss, no Focal gamma)
# ══════════════════════════════════════════════════════════════════════════════
v_cfg = _make_cfg()
v_crit = SeaIceLoss(v_cfg.train).to(device)
v_crit.mask_loss.focal   = FocalLoss(alpha=0.5, gamma=0.0)   # gamma=0 → plain BCE
v_crit.mask_loss.tversky = TverskyLoss(alpha=0.5, beta=0.5)  # balanced Dice
abl_results['BCE + Dice (baseline)'] = _short_train(
    'BCE+Dice', v_cfg, v_crit, _tr, _va, _te, device)

# ── Save ─────────────────────────────────────────────────────────────────────
with open(ABL_JSON, 'w') as f: json.dump(abl_results, f, indent=2)
try: shutil.copy(ABL_JSON, '/content/drive/MyDrive/sea_ice_seg/outputs/ablation_results.json')
except Exception: pass
print(f'\n✅ All retraining ablations done.  Results → {ABL_JSON}')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  ABLATION — Cell C: Results table + LaTeX output (full metric suite)    ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import json
from pathlib import Path

OUTPUT_DIR = globals().get('OUTPUT_DIR', '/content/sea_ice_seg/outputs')
ABL_JSON   = f'{OUTPUT_DIR}/ablation_results.json'

if not Path(ABL_JSON).exists():
    print('⚠️  ablation_results.json not found. Run Cells A and B first.')
else:
    with open(ABL_JSON) as f:
        results = json.load(f)

    GROUPS = {
        'Full Model': ['Full Model (Ours)'],
        'Component Ablations': [
            'w/o Temporal Consistency',
            'w/o Depth Features',
            'w/o LoRA (Frozen CLIP)',
            'Token Decoder',
            'w/o Auxiliary Loss',
        ],
        'Loss Function Ablations': [
            'Focal Loss Only',
            'Tversky Loss Only',
            'BCE + Dice (baseline)',
        ],
    }
    # (display label, json key) — cIoU and Boundary IoU now included
    METRICS = [('mIoU', 'mean_iou'), ('cIoU', 'ciou'),
               ('BIoU', 'boundary_iou'), ('Dice', 'mean_dice'),
               ('F1', 'weighted_f1'), ('PxAcc', 'pixel_accuracy')]

    def _g(r, k):
        v = r.get(k, None)
        return float(v) if v is not None else float('nan')

    # ── Console table ─────────────────────────────────────────────────────────
    W = 36
    hdr = f'{"Variant":<{W}}' + ''.join(f'{m:>8}' for m, _ in METRICS)
    print('\n' + '─' * len(hdr)); print(hdr); print('─' * len(hdr))
    for grp, names in GROUPS.items():
        print(f'  {grp}')
        for name in names:
            if name not in results: continue
            r = results[name]
            row = f'  {name:<{W-2}}' + ''.join(f'{_g(r, k):>8.4f}' for _, k in METRICS)
            print(row + ('  ★' if 'Ours' in name else ''))
    print('─' * len(hdr))

    # ── Threshold sweep ───────────────────────────────────────────────────────
    tks = [k for k in results if k.startswith('Threshold=')]
    if tks:
        print('\nThreshold Sensitivity:')
        print(f'  {"Threshold":<12}{"mIoU":>8}{"cIoU":>8}{"BIoU":>8}{"F1":>8}')
        for k in sorted(tks):
            r = results[k]; th = k.split('=')[1]
            mk = '  ← default' if th == '0.5' else ''
            print(f'  {th:<12}{_g(r,"mean_iou"):>8.4f}{_g(r,"ciou"):>8.4f}'
                  f'{_g(r,"boundary_iou"):>8.4f}{_g(r,"weighted_f1"):>8.4f}{mk}')

    # ── LaTeX table ───────────────────────────────────────────────────────────
    print('\n' + '─' * 60)
    print('LaTeX Table (copy into your paper):')
    print('─' * 60)
    L  = '\\begin{table}[t]\n\\centering\n'
    L += ('\\caption{Ablation study on the sea ice SAR reasoning segmentation '
          'pipeline. Variants are trained for up to 15 epochs with early '
          'stopping (patience=5) on the same split as the full model. '
          'gIoU is the mean per-image IoU and cIoU the cumulative IoU.}\n')
    L += '\\label{tab:ablation}\n'
    L += '\\begin{tabular}{lcccccc}\n\\toprule\n'
    L += ('Variant & mIoU$\\uparrow$ & cIoU$\\uparrow$ & B-IoU$\\uparrow$ & '
          'Dice$\\uparrow$ & F1$\\uparrow$ & PxAcc$\\uparrow$ \\\\\n\\midrule\n')
    for grp, names in GROUPS.items():
        L += f'\\multicolumn{{7}}{{l}}{{\\textit{{{grp}}}}} \\\\\n'
        for name in names:
            if name not in results: continue
            r = results[name]
            vals = ' & '.join(f'{_g(r, k):.4f}' for _, k in METRICS)
            nm = name.replace('&', '\\&')
            if 'Ours' in name:
                cells_tex = ' & '.join(f'\\textbf{{{_g(r, k):.4f}}}' for _, k in METRICS)
                L += f'\\quad \\textbf{{{nm}}} & {cells_tex} \\\\\n'
            else:
                L += f'\\quad {nm} & {vals} \\\\\n'
    L += '\\bottomrule\n\\end{tabular}\n\\end{table}'
    print(L)
    print('\n✅ Table printed above — copy the LaTeX block into your paper.')


## 19. 💾 Final Save to Google Drive

In [ ]:
import os, shutil
from pathlib import Path

print('Syncing all outputs to Google Drive...')

# Copy everything in OUTPUT_DIR to Drive
output_path = Path(OUTPUT_DIR)
drive_path  = Path(DRIVE_OUTPUT)
drive_path.mkdir(parents=True, exist_ok=True)

for f in output_path.rglob('*'):
    if f.is_file():
        rel = f.relative_to(output_path)
        dest = drive_path / rel
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, dest)

print('\nFiles saved to Drive:')
for f in sorted(drive_path.rglob('*')):
    if f.is_file():
        size = f.stat().st_size / 1e6
        print(f'  {f.relative_to(drive_path)}  ({size:.1f} MB)')

print(f'\n✅ All outputs synced to: {DRIVE_OUTPUT}')
print('\nTo resume training in a new session:')
print(f'  Set RESUME_CHECKPOINT = "{DRIVE_OUTPUT}/best_model.pth"')
print('  in Cell 12 (Training Setup) before running training.')

---
## 📝 Tips & Troubleshooting

| Issue | Fix |
|-------|-----|
| **CUDA OOM** | Reduce `BATCH_SIZE` to 1, or set `IMAGE_SIZE = (384, 384)` in Cell 5 |
| **Session disconnects** | Drive auto-saves checkpoints; resume by setting `RESUME_CHECKPOINT` in Cell 12 |
| **Slow data loading** | Set `NUM_WORKERS = 0` if you see hangs |
| **W&B logging** | Set `USE_WANDB = True` and run `!wandb login` before Cell 12 |
| **Better accuracy** | Increase `DECODER_BASE` (U-Net width) on A100. Do NOT enable SAM — it is frozen and bypasses the trainable U-Net decoder |
| **LLM-guided reasoning** | Requires `llm_backend = 'blip2'` — only on A100 (needs ~12 GB extra) |

### Key Paths
```
Repo          : /content/sea_ice_seg/
Dataset       : /content/sea_ice_seg/dataset/
Best model    : /content/sea_ice_seg/outputs/best_model.pth
Drive backup  : /content/drive/MyDrive/sea_ice_seg/outputs/
```